In [1]:
# Whole Missingness
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt 
import seaborn as sns

import json
import glob
from pathlib import Path

In [10]:
def variable_audit(df, var):

    print("Variable:", var)
    print("-" * 40)

    missing_count = df[var].isna().sum()
    missing_rate = df[var].isna().mean()

    print("Missing count:", missing_count)
    print("Missing rate:", round(missing_rate, 4))

    print("Dtype:", df[var].dtype)

    print()
    
    if df[var].dtype == "object" or df[var].nunique() < 20:

        print("Value counts:")
        print(df[var].value_counts(dropna=False))

    else:

        print("Summary statistics:")
        print(df[var].describe())

        print()
        print("Quantiles:")
        print(df[var].quantile([0,0.25,0.5,0.75,0.9,0.95,0.99]))

In [40]:
def missing_rate(df, var):
    entity_missing = (
        df.groupby("organisation_number")[var]
            .agg(lambda x: x.notna().any())
            .reset_index(name="A_exists")
        )
    missing_rate = 1 - entity_missing["A_exists"].mean()

    print()
    print("entity missing of", var, ":", round(missing_rate, 4))

In [3]:
with open("Charity.json", encoding="utf-8-sig") as f:
    df = json.load(f)
df = pd.json_normalize(df)

In [4]:
dfr=df[df['charity_registration_status']=='Registered']

In [20]:
dfr.info()
dfr.nunique()

<class 'pandas.core.frame.DataFrame'>
Index: 185003 entries, 1 to 395527
Data columns (total 34 columns):
 #   Column                               Non-Null Count   Dtype  
---  ------                               --------------   -----  
 0   date_of_extract                      185003 non-null  object 
 1   organisation_number                  185003 non-null  int64  
 2   registered_charity_number            185003 non-null  int64  
 3   linked_charity_number                185003 non-null  int64  
 4   charity_name                         185003 non-null  object 
 5   charity_type                         171166 non-null  object 
 6   charity_registration_status          185003 non-null  object 
 7   date_of_registration                 185003 non-null  object 
 8   date_of_removal                      0 non-null       object 
 9   charity_reporting_status             171166 non-null  object 
 10  latest_acc_fin_period_start_date     162664 non-null  object 
 11  latest_acc_fin_per

date_of_extract                             1
organisation_number                    185003
registered_charity_number              171299
linked_charity_number                     228
charity_name                           181895
charity_type                                5
charity_registration_status                 1
date_of_registration                    16687
date_of_removal                             0
charity_reporting_status                    6
latest_acc_fin_period_start_date         1700
latest_acc_fin_period_end_date           1442
latest_income                           88431
latest_expenditure                      87510
charity_contact_address1               136931
charity_contact_address2                74939
charity_contact_address3                31953
charity_contact_address4                 9547
charity_contact_address5                 3275
charity_contact_postcode               125802
charity_contact_phone                  154745
charity_contact_email             

In [22]:
with open("Charity Area of Operation.json", encoding="utf-8-sig") as f:
    dfarea = json.load(f)
dfarea = pd.json_normalize(dfarea)

In [23]:
dfrarea = dfarea[
    dfarea["organisation_number"].isin(
        dfr["organisation_number"]
    )
]

In [24]:
dfrarea.info()

<class 'pandas.core.frame.DataFrame'>
Index: 357788 entries, 5 to 535966
Data columns (total 9 columns):
 #   Column                              Non-Null Count   Dtype 
---  ------                              --------------   ----- 
 0   date_of_extract                     357788 non-null  object
 1   organisation_number                 357788 non-null  int64 
 2   registered_charity_number           357788 non-null  int64 
 3   linked_charity_number               357788 non-null  int64 
 4   geographic_area_type                357788 non-null  object
 5   geographic_area_description         357788 non-null  object
 6   parent_geographic_area_type         182656 non-null  object
 7   parent_geographic_area_description  182656 non-null  object
 8   welsh_ind                           357788 non-null  bool  
dtypes: bool(1), int64(3), object(5)
memory usage: 24.9+ MB


In [25]:
variable_audit(dfrarea, 'geographic_area_type')

Variable: geographic_area_type
----------------------------------------
Missing count: 0
Missing rate: 0.0
Dtype: object

Value counts:
geographic_area_type
Local Authority    184958
Country            127539
Region              45291
Name: count, dtype: int64


In [38]:
variable_audit(dfrarea, 'geographic_area_description')

Variable: geographic_area_description
----------------------------------------
Missing count: 0
Missing rate: 0.0
Dtype: object

Value counts:
geographic_area_description
Throughout England And Wales         27178
Throughout England                   10831
Scotland                              6902
Throughout London                     5817
Kent                                  4784
                                     ...  
Jarvis Island                           32
British Antarctic Territory             32
Heard Island And Mcdonald Islands       30
Bouvet Island                           30
Netherlands Antilles                    17
Name: count, Length: 453, dtype: int64


In [27]:
mergedarea = dfr.merge(
    dfrarea,
    on="organisation_number",
    how="left"
)
# mergedarea.info()

In [41]:
variable_audit(mergedarea, 'geographic_area_type')
missing_rate(mergedarea, 'geographic_area_type')

Variable: geographic_area_type
----------------------------------------
Missing count: 13855
Missing rate: 0.0373
Dtype: object

Value counts:
geographic_area_type
Local Authority    184958
Country            127539
Region              45291
NaN                 13855
Name: count, dtype: int64

entity missing of geographic_area_type : 0.0749


In [42]:
variable_audit(mergedarea, 'geographic_area_description')
missing_rate(mergedarea, 'geographic_area_description')

Variable: geographic_area_description
----------------------------------------
Missing count: 13855
Missing rate: 0.0373
Dtype: object

Value counts:
geographic_area_description
Throughout England And Wales         27178
NaN                                  13855
Throughout England                   10831
Scotland                              6902
Throughout London                     5817
                                     ...  
Wake Island                             32
Howland Island                          32
Heard Island And Mcdonald Islands       30
Bouvet Island                           30
Netherlands Antilles                    17
Name: count, Length: 454, dtype: int64

entity missing of geographic_area_description : 0.0749


In [68]:
with open("Charity Classification.json", encoding="utf-8-sig") as f:
    dfcla = json.load(f)
dfcla = pd.json_normalize(dfcla)

In [69]:
dfrcla = dfcla[
    dfcla["organisation_number"].isin(
        dfr["organisation_number"]
    )
]

In [70]:
dfrcla.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1145847 entries, 17 to 1729015
Data columns (total 7 columns):
 #   Column                      Non-Null Count    Dtype 
---  ------                      --------------    ----- 
 0   date_of_extract             1145847 non-null  object
 1   organisation_number         1145847 non-null  int64 
 2   registered_charity_number   1145847 non-null  int64 
 3   linked_charity_number       1145847 non-null  int64 
 4   classification_code         1145847 non-null  int64 
 5   classification_type         1145847 non-null  object
 6   classification_description  1145847 non-null  object
dtypes: int64(4), object(3)
memory usage: 69.9+ MB


In [71]:
variable_audit(dfrcla, 'classification_type')

Variable: classification_type
----------------------------------------
Missing count: 0
Missing rate: 0.0
Dtype: object

Value counts:
classification_type
What    426907
Who     368141
How     350799
Name: count, dtype: int64


In [72]:
variable_audit(dfrcla, 'classification_description')

Variable: classification_description
----------------------------------------
Missing count: 0
Missing rate: 0.0
Dtype: object

Value counts:
classification_description
Children/young People                                             95503
The General Public/mankind                                        91716
Education/training                                                83928
Provides Services                                                 72452
General Charitable Purposes                                       58819
Provides Buildings/facilities/open Space                          54577
Elderly/old People                                                51639
Makes Grants To Organisations                                     49101
Provides Advocacy/advice/information                              47839
People With Disabilities                                          45973
Other Charities Or Voluntary Bodies                               40930
Religious Activities                   

In [73]:
dfr_wide = (
    dfrcla.pivot_table(
        index="organisation_number",
        columns="classification_type",
        values="classification_description",
        aggfunc="first"
    )
    .reset_index()
)
dfrcla = dfrcla.merge(dfr_wide, on="organisation_number", how="left")

In [74]:
variable_audit(dfrcla, 'What')

Variable: What
----------------------------------------
Missing count: 83
Missing rate: 0.0001
Dtype: object

Value counts:
What
General Charitable Purposes                                       512747
Education/training                                                358525
Religious Activities                                               69050
The Advancement Of Health Or Saving Of Lives                       44229
The Prevention Or Relief Of Poverty                                31883
Disability                                                         30497
Arts/culture/heritage/science                                      26514
Amateur Sport                                                      24172
Environment/conservation/heritage                                  11422
Other Charitable Purposes                                           8694
Animals                                                             7094
Accommodation/housing                                               

In [75]:
variable_audit(dfrcla, 'Who')

Variable: Who
----------------------------------------
Missing count: 4775
Missing rate: 0.0042
Dtype: object

Value counts:
Who
Children/young People                             757809
The General Public/mankind                        164759
Other Charities Or Voluntary Bodies                64797
Elderly/old People                                 59095
Other Defined Groups                               52042
People With Disabilities                           25162
People Of A Particular Ethnic Or Racial Origin     17408
NaN                                                 4775
Name: count, dtype: int64


In [76]:
variable_audit(dfrcla, 'How')

Variable: How
----------------------------------------
Missing count: 71
Missing rate: 0.0001
Dtype: object

Value counts:
How
Makes Grants To Individuals                 280014
Provides Buildings/facilities/open Space    224499
Provides Services                           184274
Makes Grants To Organisations               163240
Provides Human Resources                    152993
Provides Other Finance                       45948
Other Charitable Activities                  41982
Provides Advocacy/advice/information         41961
Acts As An Umbrella Or Resource Body          7525
Sponsors Or Undertakes Research               3340
NaN                                             71
Name: count, dtype: int64


In [77]:
mergedcla = dfr.merge(
    dfrcla,
    on="organisation_number",
    how="left"
)
# mergedcla.info()

In [78]:
variable_audit(mergedcla, 'classification_type')
missing_rate(mergedcla, 'classification_type')

Variable: classification_type
----------------------------------------
Missing count: 13933
Missing rate: 0.012
Dtype: object

Value counts:
classification_type
What    426907
Who     368141
How     350799
NaN      13933
Name: count, dtype: int64

entity missing of classification_type : 0.0753


In [79]:
variable_audit(mergedcla, 'classification_description')
missing_rate(mergedcla, 'classification_description')

Variable: classification_description
----------------------------------------
Missing count: 13933
Missing rate: 0.012
Dtype: object

Value counts:
classification_description
Children/young People                                             95503
The General Public/mankind                                        91716
Education/training                                                83928
Provides Services                                                 72452
General Charitable Purposes                                       58819
Provides Buildings/facilities/open Space                          54577
Elderly/old People                                                51639
Makes Grants To Organisations                                     49101
Provides Advocacy/advice/information                              47839
People With Disabilities                                          45973
Other Charities Or Voluntary Bodies                               40930
Religious Activities             

In [83]:
variable_audit(mergedcla, 'What')
missing_rate(mergedcla, 'What')

Variable: What
----------------------------------------
Missing count: 14016
Missing rate: 0.0121
Dtype: object

Value counts:
What
General Charitable Purposes                                       512747
Education/training                                                358525
Religious Activities                                               69050
The Advancement Of Health Or Saving Of Lives                       44229
The Prevention Or Relief Of Poverty                                31883
Disability                                                         30497
Arts/culture/heritage/science                                      26514
Amateur Sport                                                      24172
NaN                                                                14016
Environment/conservation/heritage                                  11422
Other Charitable Purposes                                           8694
Animals                                                          

In [84]:
variable_audit(mergedcla, 'Who')
missing_rate(mergedcla, 'Who')

Variable: Who
----------------------------------------
Missing count: 18708
Missing rate: 0.0161
Dtype: object

Value counts:
Who
Children/young People                             757809
The General Public/mankind                        164759
Other Charities Or Voluntary Bodies                64797
Elderly/old People                                 59095
Other Defined Groups                               52042
People With Disabilities                           25162
NaN                                                18708
People Of A Particular Ethnic Or Racial Origin     17408
Name: count, dtype: int64

entity missing of Who : 0.0832


In [85]:
variable_audit(mergedcla, 'How')
missing_rate(mergedcla, 'How')

Variable: How
----------------------------------------
Missing count: 14004
Missing rate: 0.0121
Dtype: object

Value counts:
How
Makes Grants To Individuals                 280014
Provides Buildings/facilities/open Space    224499
Provides Services                           184274
Makes Grants To Organisations               163240
Provides Human Resources                    152993
Provides Other Finance                       45948
Other Charitable Activities                  41982
Provides Advocacy/advice/information         41961
NaN                                          14004
Acts As An Umbrella Or Resource Body          7525
Sponsors Or Undertakes Research               3340
Name: count, dtype: int64

entity missing of How : 0.0755


In [59]:
variable_audit(dfr, 'charity_activities')
missing_rate(dfr, 'charity_activities')

Variable: charity_activities
----------------------------------------
Missing count: 20929
Missing rate: 0.1131
Dtype: object

Value counts:
charity_activities
None                                                                                                                                                                                                                                                                                                                                                            20929
Religious activities                                                                                                                                                                                                                                                                                                                                              287
Religious Activities                                                                                                        

In [123]:
with open("Charity Partb.json", encoding="utf-8-sig") as f:
    dfb = json.load(f)
dfb = pd.json_normalize(dfb)

In [124]:
dfrb = dfb[
    dfb["organisation_number"].isin(
        dfr["organisation_number"]
    )
]

In [125]:
dfrb.info()

<class 'pandas.core.frame.DataFrame'>
Index: 66907 entries, 0 to 78082
Data columns (total 50 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   date_of_extract                         66907 non-null  object 
 1   organisation_number                     66907 non-null  int64  
 2   registered_charity_number               66907 non-null  int64  
 3   latest_fin_period_submitted_ind         66907 non-null  bool   
 4   fin_period_order_number                 66907 non-null  int64  
 5   ar_cycle_reference                      66907 non-null  object 
 6   fin_period_start_date                   66907 non-null  object 
 7   fin_period_end_date                     66907 non-null  object 
 8   ar_due_date                             66907 non-null  object 
 9   ar_received_date                        66907 non-null  object 
 10  income_donations_and_legacies           66903 non-null  float64

In [126]:
variable_audit(dfrb, 'income_donations_and_legacies')

Variable: income_donations_and_legacies
----------------------------------------
Missing count: 4
Missing rate: 0.0001
Dtype: float64

Summary statistics:
count        66903.00
mean       2054682.76
std       18283584.72
min              0.00
25%          18283.50
50%         323233.00
75%         930083.00
max     1303248227.00
Name: income_donations_and_legacies, dtype: float64

Quantiles:
0.00          0.00
0.25      18283.50
0.50     323233.00
0.75     930083.00
0.90    2720052.60
0.95    5646081.70
0.99   28603833.00
Name: income_donations_and_legacies, dtype: float64


In [127]:
variable_audit(dfrb, 'income_other_trading_activities')

Variable: income_other_trading_activities
----------------------------------------
Missing count: 4
Missing rate: 0.0001
Dtype: float64

Summary statistics:
count        66903.00
mean        525054.35
std        7977721.49
min              0.00
25%              0.00
50%              0.00
75%          67235.50
max     1253193000.00
Name: income_other_trading_activities, dtype: float64

Quantiles:
0.00         0.00
0.25         0.00
0.50         0.00
0.75     67235.50
0.90    561127.20
0.95   1407959.80
0.99   7190800.24
Name: income_other_trading_activities, dtype: float64


In [128]:
variable_audit(dfrb, 'income_charitable_activities')

Variable: income_charitable_activities
----------------------------------------
Missing count: 4
Missing rate: 0.0001
Dtype: float64

Summary statistics:
count        66903.00
mean       3298841.10
std       19544083.68
min         -61000.00
25%            887.00
50%         511155.00
75%        1609603.00
max     1420400000.00
Name: income_charitable_activities, dtype: float64

Quantiles:
0.00     -61000.00
0.25        887.00
0.50     511155.00
0.75    1609603.00
0.90    5744515.20
0.95   12499273.90
0.99   44983994.34
Name: income_charitable_activities, dtype: float64


In [129]:
variable_audit(dfrb, 'income_investments')

Variable: income_investments
----------------------------------------
Missing count: 4
Missing rate: 0.0001
Dtype: float64

Summary statistics:
count       66903.00
mean       319858.06
std       4279540.32
min        -21766.00
25%            61.50
50%          4930.00
75%         56120.00
max     477396888.00
Name: income_investments, dtype: float64

Quantiles:
0.00    -21766.00
0.25        61.50
0.50      4930.00
0.75     56120.00
0.90    408275.00
0.95    977074.70
0.99   4606274.94
Name: income_investments, dtype: float64


In [130]:
variable_audit(dfrb, 'income_other')

Variable: income_other
----------------------------------------
Missing count: 4
Missing rate: 0.0001
Dtype: float64

Summary statistics:
count       66903.00
mean       224811.47
std       4017242.08
min        -16532.00
25%             0.00
50%             0.00
75%         16952.50
max     433083000.00
Name: income_other, dtype: float64

Quantiles:
0.00    -16532.00
0.25         0.00
0.50         0.00
0.75     16952.50
0.90    180233.60
0.95    525000.00
0.99   2971357.30
Name: income_other, dtype: float64


In [131]:
variable_audit(dfrb, 'income_total_income_and_endowments')

Variable: income_total_income_and_endowments
----------------------------------------
Missing count: 14
Missing rate: 0.0002
Dtype: float64

Summary statistics:
count        66893.00
mean       6423397.38
std       31934018.73
min              0.00
25%         762645.00
50%        1391857.00
75%        3634147.00
max     1488655343.00
Name: income_total_income_and_endowments, dtype: float64

Quantiles:
0.00          0.00
0.25     762645.00
0.50    1391857.00
0.75    3634147.00
0.90   11164886.60
0.95   21494957.60
0.99   84344280.00
Name: income_total_income_and_endowments, dtype: float64


In [132]:
variable_audit(dfrb, 'income_legacies')

Variable: income_legacies
----------------------------------------
Missing count: 4
Missing rate: 0.0001
Dtype: float64

Summary statistics:
count       66903.00
mean       259298.49
std       3426046.27
min             0.00
25%             0.00
50%             0.00
75%             0.00
max     287624420.00
Name: income_legacies, dtype: float64

Quantiles:
0.00         0.00
0.25         0.00
0.50         0.00
0.75         0.00
0.90    184975.40
0.95    642989.60
0.99   3680960.00
Name: income_legacies, dtype: float64


In [133]:
variable_audit(dfrb, 'income_endowments')

Variable: income_endowments
----------------------------------------
Missing count: 4
Missing rate: 0.0001
Dtype: float64

Summary statistics:
count       66903.00
mean        64469.38
std       1901936.23
min             0.00
25%             0.00
50%             0.00
75%             0.00
max     250000000.00
Name: income_endowments, dtype: float64

Quantiles:
0.00        0.00
0.25        0.00
0.50        0.00
0.75        0.00
0.90        0.00
0.95        0.00
0.99   682855.72
Name: income_endowments, dtype: float64


In [134]:
mergedb = dfr.merge(
    dfrb,
    on="organisation_number",
    how="left"
)
# mergedb.info()

In [136]:
variable_audit(mergedb, 'income_donations_and_legacies')
missing_rate(mergedb, 'income_donations_and_legacies')

Variable: income_donations_and_legacies
----------------------------------------
Missing count: 166577
Missing rate: 0.7135
Dtype: float64

Summary statistics:
count        66903.00
mean       2054682.76
std       18283584.72
min              0.00
25%          18283.50
50%         323233.00
75%         930083.00
max     1303248227.00
Name: income_donations_and_legacies, dtype: float64

Quantiles:
0.00          0.00
0.25      18283.50
0.50     323233.00
0.75     930083.00
0.90    2720052.60
0.95    5646081.70
0.99   28603833.00
Name: income_donations_and_legacies, dtype: float64

entity missing of income_donations_and_legacies : 0.9004


In [137]:
variable_audit(mergedb, 'income_other_trading_activities')
missing_rate(mergedb, 'income_other_trading_activities')

Variable: income_other_trading_activities
----------------------------------------
Missing count: 166577
Missing rate: 0.7135
Dtype: float64

Summary statistics:
count        66903.00
mean        525054.35
std        7977721.49
min              0.00
25%              0.00
50%              0.00
75%          67235.50
max     1253193000.00
Name: income_other_trading_activities, dtype: float64

Quantiles:
0.00         0.00
0.25         0.00
0.50         0.00
0.75     67235.50
0.90    561127.20
0.95   1407959.80
0.99   7190800.24
Name: income_other_trading_activities, dtype: float64

entity missing of income_other_trading_activities : 0.9004


In [138]:
variable_audit(mergedb, 'income_charitable_activities')
missing_rate(mergedb, 'income_charitable_activities')

Variable: income_charitable_activities
----------------------------------------
Missing count: 166577
Missing rate: 0.7135
Dtype: float64

Summary statistics:
count        66903.00
mean       3298841.10
std       19544083.68
min         -61000.00
25%            887.00
50%         511155.00
75%        1609603.00
max     1420400000.00
Name: income_charitable_activities, dtype: float64

Quantiles:
0.00     -61000.00
0.25        887.00
0.50     511155.00
0.75    1609603.00
0.90    5744515.20
0.95   12499273.90
0.99   44983994.34
Name: income_charitable_activities, dtype: float64

entity missing of income_charitable_activities : 0.9004


In [139]:
variable_audit(mergedb, 'income_investments')
missing_rate(mergedb, 'income_investments')

Variable: income_investments
----------------------------------------
Missing count: 166577
Missing rate: 0.7135
Dtype: float64

Summary statistics:
count       66903.00
mean       319858.06
std       4279540.32
min        -21766.00
25%            61.50
50%          4930.00
75%         56120.00
max     477396888.00
Name: income_investments, dtype: float64

Quantiles:
0.00    -21766.00
0.25        61.50
0.50      4930.00
0.75     56120.00
0.90    408275.00
0.95    977074.70
0.99   4606274.94
Name: income_investments, dtype: float64

entity missing of income_investments : 0.9004


In [140]:
variable_audit(mergedb, 'income_other')
missing_rate(mergedb, 'income_other')

Variable: income_other
----------------------------------------
Missing count: 166577
Missing rate: 0.7135
Dtype: float64

Summary statistics:
count       66903.00
mean       224811.47
std       4017242.08
min        -16532.00
25%             0.00
50%             0.00
75%         16952.50
max     433083000.00
Name: income_other, dtype: float64

Quantiles:
0.00    -16532.00
0.25         0.00
0.50         0.00
0.75     16952.50
0.90    180233.60
0.95    525000.00
0.99   2971357.30
Name: income_other, dtype: float64

entity missing of income_other : 0.9004


In [141]:
variable_audit(mergedb, 'income_total_income_and_endowments')
missing_rate(mergedb, 'income_total_income_and_endowments')

Variable: income_total_income_and_endowments
----------------------------------------
Missing count: 166587
Missing rate: 0.7135
Dtype: float64

Summary statistics:
count        66893.00
mean       6423397.38
std       31934018.73
min              0.00
25%         762645.00
50%        1391857.00
75%        3634147.00
max     1488655343.00
Name: income_total_income_and_endowments, dtype: float64

Quantiles:
0.00          0.00
0.25     762645.00
0.50    1391857.00
0.75    3634147.00
0.90   11164886.60
0.95   21494957.60
0.99   84344280.00
Name: income_total_income_and_endowments, dtype: float64

entity missing of income_total_income_and_endowments : 0.9004


In [142]:
variable_audit(mergedb, 'income_endowments')
missing_rate(mergedb, 'income_endowments')

Variable: income_endowments
----------------------------------------
Missing count: 166577
Missing rate: 0.7135
Dtype: float64

Summary statistics:
count       66903.00
mean        64469.38
std       1901936.23
min             0.00
25%             0.00
50%             0.00
75%             0.00
max     250000000.00
Name: income_endowments, dtype: float64

Quantiles:
0.00        0.00
0.25        0.00
0.50        0.00
0.75        0.00
0.90        0.00
0.95        0.00
0.99   682855.72
Name: income_endowments, dtype: float64

entity missing of income_endowments : 0.9004


In [143]:
variable_audit(mergedb, 'income_legacies')
missing_rate(mergedb, 'income_legacies')

Variable: income_legacies
----------------------------------------
Missing count: 166577
Missing rate: 0.7135
Dtype: float64

Summary statistics:
count       66903.00
mean       259298.49
std       3426046.27
min             0.00
25%             0.00
50%             0.00
75%             0.00
max     287624420.00
Name: income_legacies, dtype: float64

Quantiles:
0.00         0.00
0.25         0.00
0.50         0.00
0.75         0.00
0.90    184975.40
0.95    642989.60
0.99   3680960.00
Name: income_legacies, dtype: float64

entity missing of income_legacies : 0.9004


In [92]:
income_cols = [
    "income_donations_and_legacies",
    "income_charitable_activities",
    "income_other_trading_activities",
    "income_investments",
    "income_other",
    "income_endowments"
]

dfrb["income_sum_check"] = dfrb[income_cols].sum(axis=1)

dfrb["income_diff"] = (
    dfrb["income_sum_check"] - dfrb["income_total_income_and_endowments"]
)

/var/folders/6c/2pm3cz6s4lj_z9wz3v6ndxg80000gn/T/ipykernel_11618/2986034213.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfrb["income_sum_check"] = dfrb[income_cols].sum(axis=1)
/var/folders/6c/2pm3cz6s4lj_z9wz3v6ndxg80000gn/T/ipykernel_11618/2986034213.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfrb["income_diff"] = (


In [100]:
pd.set_option('display.float_format', '{:.2f}'.format)
dfrb["income_diff"].describe()

count       66893.00
mean        64558.42
std       1902147.14
min       -579170.00
25%             0.00
50%             0.00
75%             0.00
max     250000000.00
Name: income_diff, dtype: float64

In [105]:
(abs(dfrb["income_diff"]) > 1).mean()

np.float64(0.028576979987146336)

In [96]:
(dfrb["income_legacies"] <= dfrb["income_donations_and_legacies"]).mean()

np.float64(0.9999402155230395)

In [103]:
dfrb[dfrb["income_diff"]<0]

,date_of_extract,organisation_number,registered_charity_number,latest_fin_period_submitted_ind,fin_period_order_number,ar_cycle_reference,fin_period_start_date,fin_period_end_date,ar_due_date,ar_received_date,...,funds_unrestricted,funds_restricted,funds_total,count_employees,charity_only_accounts,consolidated_accounts,income_components_sum,income_match,income_sum_check,income_diff
14,2026-02-26T00:00:00,200240,200240,True,1,AR24,2023-11-01T00:00:00,2024-10-31T00:00:00,2025-08-31T00:00:00,2025-07-28T00:00:00,...,15260417.00,4313.00,15264730.00,5.00,True,None,686821.00,False,686821.00,-40.00
7588,2026-02-26T00:00:00,246589,246589,True,1,AR25,2024-04-01T00:00:00,2025-03-31T00:00:00,2026-01-31T00:00:00,2026-01-13T00:00:00,...,14719.00,333270.00,347989.00,41.00,True,None,645096.00,False,645096.00,-32998.00
20372,2026-02-26T00:00:00,3965494,1078360,False,5,AR19,2018-09-01T00:00:00,2019-08-31T00:00:00,2020-06-30T00:00:00,2020-06-05T00:00:00,...,98325.00,0.00,98325.00,63.00,None,True,2534193.00,False,2534193.00,-20.00
31173,2026-02-26T00:00:00,1016532,1016532,False,2,AR24,2023-04-01T00:00:00,2024-03-31T00:00:00,2025-01-31T00:00:00,2025-01-24T00:00:00,...,13798000.00,255000.00,14053000.00,280.00,None,True,34213000.00,False,33000000.00,-61000.00
32667,2026-02-26T00:00:00,800987,800987,False,3,AR23,2022-04-01T00:00:00,2023-03-31T00:00:00,2024-01-31T00:00:00,2024-01-30T00:00:00,...,5126681.00,0.00,5126681.00,69.00,True,None,5237816.00,False,5237816.00,-21766.00
37261,2026-02-26T00:00:00,3963446,1077573,False,3,AR17,2016-07-01T00:00:00,2017-06-30T00:00:00,2018-04-30T00:00:00,2019-02-01T00:00:00,...,3214980.00,0.00,3214980.00,150.00,None,True,3716338.00,False,3712338.00,-1.00
43603,2026-02-26T00:00:00,3981501,1087514,False,3,AR10,2009-04-01T00:00:00,2010-03-31T00:00:00,2011-01-31T00:00:00,2011-02-08T00:00:00,...,0.00,212006.00,212006.00,19.00,True,None,700310.00,False,700310.00,-2.00
45032,2026-02-26T00:00:00,4015921,1112020,False,4,AR16,2015-04-01T00:00:00,2016-03-31T00:00:00,2017-01-31T00:00:00,2016-11-24T00:00:00,...,1163072.00,2403.00,1165475.00,166.00,True,None,3654397.00,False,3654397.00,-1.00
48514,2026-02-26T00:00:00,4021496,1113069,True,1,AR25,2024-04-01T00:00:00,2025-03-31T00:00:00,2026-01-31T00:00:00,2025-07-24T00:00:00,...,328449.00,311222.00,639671.00,11.00,True,None,591582.00,False,591582.00,-250.00
76907,2026-02-26T00:00:00,5133520,1182006,True,1,AR25,2024-04-01T00:00:00,2025-03-31T00:00:00,2026-01-31T00:00:00,2026-01-28T00:00:00,...,NaN,NaN,NaN,NaN,None,None,0.00,False,0.00,-579170.00


In [104]:
dfrb[dfrb["income_diff"]>0]

,date_of_extract,organisation_number,registered_charity_number,latest_fin_period_submitted_ind,fin_period_order_number,ar_cycle_reference,fin_period_start_date,fin_period_end_date,ar_due_date,ar_received_date,...,funds_unrestricted,funds_restricted,funds_total,count_employees,charity_only_accounts,consolidated_accounts,income_components_sum,income_match,income_sum_check,income_diff
58,2026-02-26T00:00:00,206524,206524,False,3,AR22,2022-01-01T00:00:00,2022-12-31T00:00:00,2023-10-31T00:00:00,2023-10-05T00:00:00,...,1536014.00,0.00,35563894.00,7.00,True,None,1909858.00,False,1639663.00,4900.00
59,2026-02-26T00:00:00,206524,206524,False,4,AR21,2021-01-01T00:00:00,2021-12-31T00:00:00,2022-10-31T00:00:00,2022-10-29T00:00:00,...,1571429.00,310187.00,39268018.00,5.00,None,True,2439295.00,False,1918684.00,85894.00
212,2026-02-26T00:00:00,212808,212808,False,4,AR21,2020-07-01T00:00:00,2021-06-30T00:00:00,2022-04-30T00:00:00,2022-06-29T00:00:00,...,98591000.00,5288000.00,145091000.00,225.00,None,True,27227000.00,False,26926000.00,6000.00
213,2026-02-26T00:00:00,212808,212808,False,3,AR22,2021-07-01T00:00:00,2022-06-30T00:00:00,2023-04-30T00:00:00,2023-03-24T00:00:00,...,93565000.00,4748000.00,137487000.00,226.00,None,True,29397000.00,False,28559000.00,475000.00
261,2026-02-26T00:00:00,214779,214779,False,5,AR21,2020-04-01T00:00:00,2021-03-31T00:00:00,2022-01-31T00:00:00,2022-01-31T00:00:00,...,220221000.00,468708000.00,706481000.00,2854.00,None,True,286624000.00,False,238238000.00,225000.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
77956,2026-02-26T00:00:00,5185107,1196705,True,1,AR25,2024-04-01T00:00:00,2025-03-31T00:00:00,2026-01-31T00:00:00,2025-12-12T00:00:00,...,26099328.00,365245.00,26464573.00,116.00,None,True,6308426.00,False,6285533.00,1600000.00
77957,2026-02-26T00:00:00,5185107,1196705,False,2,AR24,2023-04-01T00:00:00,2024-03-31T00:00:00,2025-01-31T00:00:00,2024-12-02T00:00:00,...,26579340.00,238688.00,26818028.00,96.00,None,True,5485113.00,False,5485113.00,1600000.00
77958,2026-02-26T00:00:00,5185107,1196705,False,3,AR23,2022-01-01T00:00:00,2023-03-31T00:00:00,2024-01-31T00:00:00,2023-10-30T00:00:00,...,27126440.00,8879.00,27135319.00,67.00,None,True,5689140.00,False,5689140.00,1750000.00
78027,2026-02-26T00:00:00,5201502,1202580,False,2,AR24,2023-03-31T00:00:00,2024-03-31T00:00:00,2025-01-31T00:00:00,2024-12-16T00:00:00,...,850847.00,149824.00,1000671.00,21.00,True,None,1410420.00,False,1331138.00,100000.00


In [144]:
variable_audit(dfr, 'date_of_registration')
missing_rate(dfr, 'date_of_registration')

Variable: date_of_registration
----------------------------------------
Missing count: 0
Missing rate: 0.0
Dtype: object

Value counts:
date_of_registration
1962-09-22T00:00:00    1136
1961-12-22T00:00:00     272
1963-11-21T00:00:00     157
1964-01-09T00:00:00     109
1965-11-09T00:00:00     103
                       ... 
1962-10-06T00:00:00       1
1985-11-14T00:00:00       1
1986-09-10T00:00:00       1
1985-09-05T00:00:00       1
1991-04-05T00:00:00       1
Name: count, Length: 16687, dtype: int64

entity missing of date_of_registration : 0.0


In [145]:
variable_audit(dfr, 'latest_income')
missing_rate(dfr, 'latest_income')

Variable: latest_income
----------------------------------------
Missing count: 22339
Missing rate: 0.1207
Dtype: float64

Summary statistics:
count       162664.00
mean        651270.41
std        9846236.04
min           -362.00
25%           4270.75
50%          20664.00
75%         110408.50
max     1453100000.00
Name: latest_income, dtype: float64

Quantiles:
0.00       -362.00
0.25       4270.75
0.50      20664.00
0.75     110408.50
0.90     452004.30
0.95    1242320.70
0.99   10472092.49
Name: latest_income, dtype: float64

entity missing of latest_income : 0.1207


In [168]:
variable_audit(dfr, 'latest_expenditure')
missing_rate(dfr, 'latest_expenditure')

Variable: latest_expenditure
----------------------------------------
Missing count: 22339
Missing rate: 0.1207
Dtype: float64

Summary statistics:
count       162664.00
mean        650947.45
std       10624819.68
min              0.00
25%           3906.75
50%          19915.00
75%         105028.50
max     1626627348.00
Name: latest_expenditure, dtype: float64

Quantiles:
0.00          0.00
0.25       3906.75
0.50      19915.00
0.75     105028.50
0.90     433448.10
0.95    1203362.00
0.99   10303361.66
Name: latest_expenditure, dtype: float64

entity missing of latest_expenditure : 0.1207


In [146]:
variable_audit(dfrb, 'count_employees')

Variable: count_employees
----------------------------------------
Missing count: 2
Missing rate: 0.0
Dtype: float64

Summary statistics:
count    66905.00
mean        85.05
std        730.06
min          0.00
25%          7.00
50%         21.00
75%         55.00
max     146000.00
Name: count_employees, dtype: float64

Quantiles:
0.00     0.00
0.25     7.00
0.50    21.00
0.75    55.00
0.90   155.00
0.95   283.00
0.99   956.92
Name: count_employees, dtype: float64


In [148]:
variable_audit(mergedb, 'count_employees')
missing_rate(mergedb, 'count_employees')

Variable: count_employees
----------------------------------------
Missing count: 166575
Missing rate: 0.7134
Dtype: float64

Summary statistics:
count    66905.00
mean        85.05
std        730.06
min          0.00
25%          7.00
50%         21.00
75%         55.00
max     146000.00
Name: count_employees, dtype: float64

Quantiles:
0.00     0.00
0.25     7.00
0.50    21.00
0.75    55.00
0.90   155.00
0.95   283.00
0.99   956.92
Name: count_employees, dtype: float64

entity missing of count_employees : 0.9004


In [149]:
variable_audit(dfrb, 'assets_cash')

Variable: assets_cash
----------------------------------------
Missing count: 3
Missing rate: 0.0
Dtype: float64

Summary statistics:
count        66904.00
mean       2198396.65
std       14725728.06
min              0.00
25%         251907.00
50%         573198.50
75%        1403668.75
max     1380457000.00
Name: assets_cash, dtype: float64

Quantiles:
0.00          0.00
0.25     251907.00
0.50     573198.50
0.75    1403668.75
0.90    3974599.10
0.95    7446090.95
0.99   25168850.00
Name: assets_cash, dtype: float64


In [150]:
variable_audit(mergedb, 'assets_cash')
missing_rate(mergedb, 'assets_cash')

Variable: assets_cash
----------------------------------------
Missing count: 166576
Missing rate: 0.7134
Dtype: float64

Summary statistics:
count        66904.00
mean       2198396.65
std       14725728.06
min              0.00
25%         251907.00
50%         573198.50
75%        1403668.75
max     1380457000.00
Name: assets_cash, dtype: float64

Quantiles:
0.00          0.00
0.25     251907.00
0.50     573198.50
0.75    1403668.75
0.90    3974599.10
0.95    7446090.95
0.99   25168850.00
Name: assets_cash, dtype: float64

entity missing of assets_cash : 0.9004


In [151]:
variable_audit(dfrb, 'creditors_one_year_total_current')

Variable: creditors_one_year_total_current
----------------------------------------
Missing count: 3
Missing rate: 0.0
Dtype: float64

Summary statistics:
count        66904.00
mean       1904545.29
std       21209086.85
min              0.00
25%          50536.75
50%         176103.00
75%         668878.75
max     1565293891.00
Name: creditors_one_year_total_current, dtype: float64

Quantiles:
0.00          0.00
0.25      50536.75
0.50     176103.00
0.75     668878.75
0.90    2579787.60
0.95    5590943.30
0.99   23352691.65
Name: creditors_one_year_total_current, dtype: float64


In [152]:
variable_audit(mergedb, 'creditors_one_year_total_current')
missing_rate(mergedb, 'creditors_one_year_total_current')

Variable: creditors_one_year_total_current
----------------------------------------
Missing count: 166576
Missing rate: 0.7134
Dtype: float64

Summary statistics:
count        66904.00
mean       1904545.29
std       21209086.85
min              0.00
25%          50536.75
50%         176103.00
75%         668878.75
max     1565293891.00
Name: creditors_one_year_total_current, dtype: float64

Quantiles:
0.00          0.00
0.25      50536.75
0.50     176103.00
0.75     668878.75
0.90    2579787.60
0.95    5590943.30
0.99   23352691.65
Name: creditors_one_year_total_current, dtype: float64

entity missing of creditors_one_year_total_current : 0.9004


In [158]:
with open("Charity Published Report.json", encoding="utf-8-sig") as f:
    dfpu = json.load(f)
dfpu = pd.json_normalize(dfpu)

In [159]:
dfrpu = dfpu[
    dfpu["organisation_number"].isin(
        dfr["organisation_number"]
    )
]

In [160]:
dfrpu.info()

<class 'pandas.core.frame.DataFrame'>
Index: 176 entries, 1 to 197
Data columns (total 7 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   date_of_extract            176 non-null    object
 1   organisation_number        176 non-null    int64 
 2   registered_charity_number  176 non-null    int64 
 3   linked_charity_number      176 non-null    int64 
 4   report_name                176 non-null    object
 5   report_location            176 non-null    object
 6   date_published             176 non-null    object
dtypes: int64(3), object(4)
memory usage: 11.0+ KB


In [161]:
variable_audit(dfrpu, 'report_name')

Variable: report_name
----------------------------------------
Missing count: 0
Missing rate: 0.0
Dtype: object

Value counts:
report_name
STATEMENT OF INQUIRY                     104
OFFICIAL WARNING                          26
INQUIRY REPORT (SORI)                     23
INTERIM MANAGER                           19
REGULATORY CASE REPORT                     2
PUBLIC NOTICE: INTENTION TO USE POWER      2
Name: count, dtype: int64


In [162]:
mergedpu = dfr.merge(
    dfrpu,
    on="organisation_number",
    how="left"
)
# mergedpu.info()

In [163]:
variable_audit(mergedpu, 'report_name')
missing_rate(mergedpu, 'report_name')

Variable: report_name
----------------------------------------
Missing count: 184841
Missing rate: 0.999
Dtype: object

Value counts:
report_name
NaN                                      184841
STATEMENT OF INQUIRY                        104
OFFICIAL WARNING                             26
INQUIRY REPORT (SORI)                        23
INTERIM MANAGER                              19
REGULATORY CASE REPORT                        2
PUBLIC NOTICE: INTENTION TO USE POWER         2
Name: count, dtype: int64

entity missing of report_name : 0.9991


In [164]:
variable_audit(dfr, 'charity_reporting_status')
missing_rate(dfr, 'charity_reporting_status')

Variable: charity_reporting_status
----------------------------------------
Missing count: 13837
Missing rate: 0.0748
Dtype: object

Value counts:
charity_reporting_status
Submission Received          143831
None                          13837
New                            7541
Submission Double Default      7468
Submission Received Late       6129
Submission Overdue             5806
Suppressed                      391
Name: count, dtype: int64

entity missing of charity_reporting_status : 0.0748


In [165]:
variable_audit(dfr, 'charity_insolvent')
missing_rate(dfr, 'charity_insolvent')

Variable: charity_insolvent
----------------------------------------
Missing count: 0
Missing rate: 0.0
Dtype: bool

Value counts:
charity_insolvent
False    184822
True        181
Name: count, dtype: int64

entity missing of charity_insolvent : 0.0


In [166]:
variable_audit(dfr, 'charity_in_administration')
missing_rate(dfr, 'charity_in_administration')

Variable: charity_in_administration
----------------------------------------
Missing count: 0
Missing rate: 0.0
Dtype: bool

Value counts:
charity_in_administration
False    184876
True        127
Name: count, dtype: int64

entity missing of charity_in_administration : 0.0
